# Phase 2: Advanced Modeling & Feature Engineering

---

## Phase 2 Overview

Phase 1 established that weather features alone are weak predictors of trip duration ($R^2$ ≈ 0.005), but confirmed that non-linear models outperform linear ones. Phase 2 builds on these findings by expanding our feature space, deploying advanced supervised models, and applying unsupervised learning to discover natural rider segments.

### Objectives

**1. Feature Engineering & Selection:** Expand beyond raw TEMP and PRECIP_AMOUNT by creating interaction terms, polynomial features, and temporal encodings (hour-of-day, day-of-week, cyclical sin/cos transforms). Apply feature selection methods (filter, wrapper, embedded) to identify the most predictive subset and compare model performance before and after engineering.

**2. Advanced Supervised Learning:** Implement two ensemble models that directly address Phase 1's limitations:
- **Random Forest Regressor** (Gorden): An ensemble of independent decision trees that reduces the overfitting risk of our single Decision Tree baseline while preserving its ability to capture threshold effects.
- **XGBoost Regressor** (Omar): A sequential boosting approach where each tree corrects the errors of the previous one, designed to squeeze maximum predictive power from structured tabular data.

Both models will undergo hyperparameter tuning and be evaluated against the Phase 1 baselines using RMSE, MAE, and $R^2$ on the validation set.

**3. Unsupervised Learning:** Apply PCA for dimensionality reduction, then use clustering (K-Means) to test our hypothesis that Bixi trips naturally segment into distinct behavioral profiles (e.g., "Utility Riders" vs. "Leisure Riders") based on environmental context and trip characteristics.

**4. Interpretation:** Extract feature importances, generate partial dependence plots, and relate findings back to our original research questions.

### Division of Labour - Phase 2

| Name | Student ID | Contribution |
|------|-----------|--------------|
| Gorden | 40263250 | Random Forest implementation, hyperparameter tuning, feature importance analysis |
| Omar Benjelloun | 40215107 | Model 2, feature engineering, unsupervised learning (PCA + clustering) |


---

# 1. Advanced Supervised Learning models (Before Feature Engineering)

## Why Random Forest Regressor?

Our Phase 1 results showed that the Decision Tree outperformed Linear Regression across all metrics, confirming that trip duration responds to non-linear thresholds rather than a simple linear slope. However, the single Decision Tree was limited to `max_depth=5` which was used to prevent overfitting on 11M+ rows. This restricted its ability to capture deeper patterns in the data.

The Random Forest will directly address this limitation. In fact, it builds many independent Decision Trees, each trained on a random subset of the data and a random subset of features which then averages their predictions. This "ensemble averaging" cancels out the individual noise of each tree, which means we can safely allow deeper trees (`max_depth=15`) without the overfitting risk that forced us to cap depth in Phase 1 (Breiman, 2001).

There are three specific reasons this model fits our problem:

1. **Threshold Handling:** Our EDA revealed that cycling behavior shifts at specific environmental boundaries (around 0 degrees C, when precipitation begins and when windchill drops below -15 degrees C). Like the Decision Tree, Random Forest captures these threshold effects naturally through its splitting mechanism, but does so more robustly by aggregating hundreds of trees.

2. **Feature Importance:** Random Forest provides a built-in measure of how much each feature contributes to predictions. This directly addresses our research question mentioned in phase 1: "to what degree do environmental fluctuations dictate trip duration?" by quantifying each weather variable's relative predictive power.

3. **Scalability:** With `n_jobs=-1`, Random Forest parallelizes across CPU cores. This will make it practical for our 11M+ row dataset. It also requires minimal preprocessing (no scaling, no normality assumptions), which simplifies the pipeline.

We therefore expect Random Forest to improve over the Phase 1 Decision Tree baseline, with the largest gains coming after feature engineering when the model has access to temporal and interaction features that capture commuting patterns alongside weather effects.

**Reference:** Breiman, L. (2001). Random Forests. *Machine Learning*, 45(1), 5–32.

## 1.1 Random Forest

### 1.1.1 Data Loading and Setup

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

# Load the processed dataset from Phase 1
df = pd.read_csv('../data/processed/bixi_weather_2025.csv')
print(f"Dataset loaded: {df.shape[0]:,} rows, {df.shape[1]} columns")
print(f"\nColumns: {list(df.columns)}")
df.head()

Dataset loaded: 11,546,527 rows, 17 columns

Columns: ['STARTSTATIONNAME', 'STARTSTATIONARRONDISSEMENT', 'STARTSTATIONLATITUDE', 'STARTSTATIONLONGITUDE', 'ENDSTATIONNAME', 'ENDSTATIONARRONDISSEMENT', 'ENDSTATIONLATITUDE', 'ENDSTATIONLONGITUDE', 'STARTTIMEMS', 'ENDTIMEMS', 'start_dt', 'end_dt', 'hour_rounded', 'duration_min', 'TEMP', 'PRECIP_AMOUNT', 'WINDCHILL']


,STARTSTATIONNAME,STARTSTATIONARRONDISSEMENT,STARTSTATIONLATITUDE,STARTSTATIONLONGITUDE,ENDSTATIONNAME,ENDSTATIONARRONDISSEMENT,ENDSTATIONLATITUDE,ENDSTATIONLONGITUDE,STARTTIMEMS,ENDTIMEMS,start_dt,end_dt,hour_rounded,duration_min,TEMP,PRECIP_AMOUNT,WINDCHILL
0,St-Viateur / Casgrain,Le Plateau-Mont-Royal,45.527013,-73.597973,Parc Molson (d'Iberville / Beaubien),Rosemont - La Petite-Patrie,45.549022,-73.591982,1767118845159,1.767120e+12,2025-12-30 13:20:45.159,2025-12-30 13:38:47.367,2025-12-30 13:00:00,18.036800,-12.7,0.0,-23.0
1,9e avenue / Masson,Rosemont - La Petite-Patrie,45.549490,-73.573272,Parc Molson (d'Iberville / Beaubien),Rosemont - La Petite-Patrie,45.549022,-73.591982,1767146967447,1.767148e+12,2025-12-30 21:09:27.447,2025-12-30 21:27:34.129,2025-12-30 21:00:00,18.111367,-14.3,0.0,-25.0
2,St-Urbain / Laurier,Le Plateau-Mont-Royal,45.521711,-73.593743,Parc Molson (d'Iberville / Beaubien),Rosemont - La Petite-Patrie,45.549022,-73.591982,1766867149246,1.766868e+12,2025-12-27 15:25:49.246,2025-12-27 15:44:35.435,2025-12-27 15:00:00,18.769817,-9.0,0.0,-14.0
3,Parc de Turin (de Lanaudière / Jean-Talon),Villeray—Saint-Michel—Parc-Extension,45.545350,-73.610330,Parc Molson (d'Iberville / Beaubien),Rosemont - La Petite-Patrie,45.549022,-73.591982,1767227508567,1.767228e+12,2025-12-31 19:31:48.567,2025-12-31 19:43:29.866,2025-12-31 20:00:00,11.688317,-8.8,0.0,-11.0
4,Parthenais / du Mont-Royal,Le Plateau-Mont-Royal,45.536404,-73.571413,Parc Molson (d'Iberville / Beaubien),Rosemont - La Petite-Patrie,45.549022,-73.591982,1766953764324,1.766955e+12,2025-12-28 15:29:24.324,2025-12-28 15:46:38.356,2025-12-28 15:00:00,17.233867,-7.0,0.0,-13.0


### 1.1.2 Phase 1 Split

In [3]:
# Same 2 features from Phase 1
baseline_features = ['TEMP', 'PRECIP_AMOUNT']
X_baseline = df[baseline_features]
y = df['duration_min']

# Same 70/15/15 split with same seed
X_train_b, X_temp_b, y_train, y_temp = train_test_split(
    X_baseline, y, test_size=0.30, random_state=RANDOM_SEED
)
X_val_b, X_test_b, y_val, y_test = train_test_split(
    X_temp_b, y_temp, test_size=0.50, random_state=RANDOM_SEED
)

print(f"Training:   {len(X_train_b):,}")
print(f"Validation: {len(X_val_b):,}")
print(f"Test:       {len(X_test_b):,}")

Training:   8,082,568
Validation: 1,731,979
Test:       1,731,980


### 1.1.3 Random Forest Model Implementation

In [4]:
# Evaluation helper (same as Phase 1)
def evaluate_model(model, X_set, y_true):
    preds = model.predict(X_set)
    rmse = np.sqrt(mean_squared_error(y_true, preds))
    mae = mean_absolute_error(y_true, preds)
    r2 = r2_score(y_true, preds)
    return [rmse, mae, r2]

# Random Forest with reasonable defaults for a large dataset
# n_estimators=100: builds 100 trees and averages them
# max_depth=15: deeper than Phase 1's Decision Tree (max_depth=5) because the ensemble averaging reduces overfitting risk
# n_jobs=-1: uses all CPU cores (important with 11M+ rows)
rf_baseline = RandomForestRegressor(
    n_estimators=100,
    max_depth=15,
    random_state=RANDOM_SEED,
    n_jobs=-1
)

print("Training Random Forest on baseline features (TEMP, PRECIP_AMOUNT)...")
rf_baseline.fit(X_train_b, y_train)
print("Training complete.")

# Evaluate on validation set
rf_baseline_results = evaluate_model(rf_baseline, X_val_b, y_val)

# Compare with the Phase 1 results 
results_before = pd.DataFrame({
    "Metric": ["RMSE (Minutes)", "MAE (Minutes)", "R² Score"],
    "Linear Regression (Phase 1)": [13.1983, 8.4000, 0.0037],  
    "Decision Tree (Phase 1)": [13.1882, 8.3873, 0.0052],
    "Random Forest (Phase 2)": rf_baseline_results
}).set_index("Metric")

display(results_before.round(4))

Training Random Forest on baseline features (TEMP, PRECIP_AMOUNT)...
Training complete.


,Linear Regression (Phase 1),Decision Tree (Phase 1),Random Forest (Phase 2)
Metric,,,
RMSE (Minutes),13.1983,13.1882,13.1808
MAE (Minutes),8.4000,8.3873,8.3800
R² Score,0.0037,0.0052,0.0063


---
# 2. Feature Engineering + New Results of Models with New Features

## 2.1 New Features

In [ ]:
# new dataframe so that we keep the original
df_eng = df.copy()

# Temporal features
# hour_rounded should already exist from Phase 1 wrangling
# will convert to date_time if it is a string
if 'hour_rounded' in df_eng.columns:
    df_eng['hour_rounded'] = pd.to_datetime(df_eng['hour_rounded'])
    df_eng['hour'] = df_eng['hour_rounded'].dt.hour
    df_eng['day_of_week'] = df_eng['hour_rounded'].dt.dayofweek  # 0=Monday, 6=Sunday
    df_eng['month'] = df_eng['hour_rounded'].dt.month
    df_eng['is_weekend'] = (df_eng['day_of_week'] >= 5).astype(int)

# Cyclical Encoding
# since 23:00 and 00:00 are close, we will use circular encoding so that they are closer than 23 and 0
df_eng['hour_sin'] = np.sin(2 * np.pi * df_eng['hour'] / 24)
df_eng['hour_cos'] = np.cos(2 * np.pi * df_eng['hour'] / 24)
df_eng['month_sin'] = np.sin(2 * np.pi * df_eng['month'] / 12)
df_eng['month_cos'] = np.cos(2 * np.pi * df_eng['month'] / 12)

# Interaction Features
# Temperature might affect duration differently on weekends vs weekdays
df_eng['temp_weekend'] = df_eng['TEMP'] * df_eng['is_weekend']
# Rain during rush hour vs rain on a weekend afternoon are different situations
df_eng['precip_rush_hour'] = df_eng['PRECIP_AMOUNT'] * df_eng['hour']

# Polynomial Features
# Temperature might have a non-linear "sweet spot", so we curve the linear plot by squaring it
df_eng['temp_squared'] = df_eng['TEMP'] ** 2

# Domain-Specific Features
# Rush hour indicator: captures commuter vs leisure behavior. we defined 7, 8, 9, 16, 17, 18 as rush hours (0=no, 1= yes)
df_eng['is_rush_hour'] = df_eng['hour'].isin([7, 8, 9, 16, 17, 18]).astype(int)

print(f"New columns added: {df_eng.shape[1] - df.shape[1]}")
print(f"Total columns: {df_eng.shape[1]}")
print(f"\nNew features: {[c for c in df_eng.columns if c not in df.columns]}")
df_eng.head(5)



New columns added: 12
Total columns: 29

New features: ['hour', 'day_of_week', 'month', 'is_weekend', 'hour_sin', 'hour_cos', 'month_sin', 'month_cos', 'temp_weekend', 'precip_rush_hour', 'temp_squared', 'is_rush_hour']


,STARTSTATIONNAME,STARTSTATIONARRONDISSEMENT,STARTSTATIONLATITUDE,STARTSTATIONLONGITUDE,ENDSTATIONNAME,ENDSTATIONARRONDISSEMENT,ENDSTATIONLATITUDE,ENDSTATIONLONGITUDE,STARTTIMEMS,ENDTIMEMS,...,month,is_weekend,hour_sin,hour_cos,month_sin,month_cos,temp_weekend,precip_rush_hour,temp_squared,is_rush_hour
0,St-Viateur / Casgrain,Le Plateau-Mont-Royal,45.527013,-73.597973,Parc Molson (d'Iberville / Beaubien),Rosemont - La Petite-Patrie,45.549022,-73.591982,1767118845159,1.767120e+12,...,12,0,-0.258819,-0.965926,-2.449294e-16,1.0,-0.0,0.0,161.29,0
1,9e avenue / Masson,Rosemont - La Petite-Patrie,45.549490,-73.573272,Parc Molson (d'Iberville / Beaubien),Rosemont - La Petite-Patrie,45.549022,-73.591982,1767146967447,1.767148e+12,...,12,0,-0.707107,0.707107,-2.449294e-16,1.0,-0.0,0.0,204.49,0
2,St-Urbain / Laurier,Le Plateau-Mont-Royal,45.521711,-73.593743,Parc Molson (d'Iberville / Beaubien),Rosemont - La Petite-Patrie,45.549022,-73.591982,1766867149246,1.766868e+12,...,12,1,-0.707107,-0.707107,-2.449294e-16,1.0,-9.0,0.0,81.00,0
3,Parc de Turin (de Lanaudière / Jean-Talon),Villeray—Saint-Michel—Parc-Extension,45.545350,-73.610330,Parc Molson (d'Iberville / Beaubien),Rosemont - La Petite-Patrie,45.549022,-73.591982,1767227508567,1.767228e+12,...,12,0,-0.866025,0.500000,-2.449294e-16,1.0,-0.0,0.0,77.44,0
4,Parthenais / du Mont-Royal,Le Plateau-Mont-Royal,45.536404,-73.571413,Parc Molson (d'Iberville / Beaubien),Rosemont - La Petite-Patrie,45.549022,-73.591982,1766953764324,1.766955e+12,...,12,1,-0.707107,-0.707107,-2.449294e-16,1.0,-7.0,0.0,49.00,0


## 2.2 Feature Selection

## 2.3 Random Forest After Feature Engineering

---
# 3. Unsupervised Learning

---
# 4. Interpretation